# Latent-Variable Models: VAEs and GANs


Sources: DS542 Gardos; Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*
(UDL, chapters 15 and 17; figures CC-BY-NC-ND, © MIT Press unless another source is credited);
*[Rocca, "Understanding Variational Autoencoders (VAEs)", 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73)*;
[Kingma and Welling, "Auto-Encoding Variational Bayes", 2013](https://arxiv.org/abs/1312.6114);
[Goodfellow et al., "Generative Adversarial Nets", 2014](https://arxiv.org/abs/1406.2661);
and other content cited on each slide.
Code cells adapted from the UDL notebooks (MIT license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).

This is the first of three lectures on generative models. It introduces what a
generative model is, then covers the two latent-variable families that the
rest of the unit builds on: variational autoencoders, whose evidence lower
bound and reparameterization trick reappear in lecture 18 on diffusion, and
generative adversarial networks, whose adversarial loss survives today as a
training tool inside the autoencoders that diffusion models run on. The full
GAN story (DCGAN, progressive growing, StyleGAN, the GAN zoo) is in module M2.


# Generative modelling

Everything up to now has been *discriminative*: given $\mathbf{x}$, predict
$y$. Lectures 10 to 14 were the one exception, since a decoder-only
transformer is a generative model of text. This unit asks the same question of
images and other data: can we learn the distribution the data came from, and
draw new samples from it?


## From discriminative to generative

**Supervised, discriminative** (lectures 1 to 9)

**Data**: $(\mathbf{x}, y)$, $\mathbf{x}$ is data, $y$ is a label

**Goal**: learn a function $\mathbf{x} \rightarrow y$

**Applications**: classification, regression, object detection, segmentation


**Unsupervised, generative** (this unit)

**Data**: $\mathbf{x}$, no labels (or labels that are part of the data)

**Goal**: learn the structure of the data, $\mathrm{Pr}(\mathbf{x})$

**Applications**: generate new examples, denoise, interpolate, compress,
detect outliers


Exceptions so far: transformers pretrained *without labels* (lecture 13), and the
decoder that *generated* text one token at a time. Now we do the same for
images and other data.


<small>Adapted from © Alexander Amini and Ava Amini, MIT 6.S191: Introduction to Deep Learning, [IntroToDeepLearning.com](http://introtodeeplearning.com)</small>

## Two things a generative model can do

Take training samples $\mathbf{x}_i \sim \mathrm{Pr}_{\text{data}}(\mathbf{x})$
and learn a model $\mathrm{Pr}_{\text{model}}(\mathbf{x})$ that represents that distribution.

**Assign a likelihood** (probabilistic model)

![](figs/17-VAEs-and-GANs/density-estimation.png)

$$
\hat{\boldsymbol{\phi}} = \underset{\boldsymbol{\phi}}{\mathrm{argmax}} \left[ \sum_{i=1}^{I} \log\big[\mathrm{Pr}(\mathbf{x}_i \mid \boldsymbol{\phi})\big] \right]
$$


**Draw samples**

![](figs/17-VAEs-and-GANs/generated-samples.png)


A model can do one, the other, or both. Maximum likelihood (lecture 3) is the
recipe for the first; the second needs a way to *produce* an $\mathbf{x}$.

<small>Figures © Alexander Amini and Ava Amini, MIT 6.S191: Introduction to Deep Learning, [IntroToDeepLearning.com](http://introtodeeplearning.com)</small>

## A taxonomy of generative models

![](figs/17-VAEs-and-GANs/unsupervised-model-taxonomy.png)


**Generative** = can generate new examples

**Probabilistic** = can assign a probability to an example

**Latent variable** = maps a random variable $\mathbf{z}$ to an example $\mathbf{x}$

| Family | P | LV |
|:--|:-:|:-:|
| Generative adversarial networks | | ✓ |
| Variational autoencoders | ✓ | ✓ |
| Diffusion models (lecture 18) | ✓ | ✓ |
| Normalizing flows | ✓ | ✓ |
| Autoregressive models (GPT) | ✓ | |
| Energy models | ✓ | |


The decoder-only transformer is the autoregressive row: it factorizes
$\mathrm{Pr}(t_1, \ldots, t_N) = \mathrm{Pr}(t_1) \prod_{n=2}^{N} \mathrm{Pr}(t_n \mid t_1, \ldots, t_{n-1})$,
so it can score a sequence and generate one, but it has no latent variable.
This lecture is about the two latent-variable families in the table that are not
diffusion: VAEs, which are probabilistic, and GANs, which are not.


## Latent-variable models and how to fit them

![A latent-variable model maps a random "latent" variable to a new data sample.](figs/17-VAEs-and-GANs/latent-variable-model-pipeline.png)


![(a) Fitting a GAN: make generated samples indistinguishable from real ones. (b) Fitting a VAE, flow or diffusion model: make the real samples likely under the model's density. UDL ch. 15.](figs/17-VAEs-and-GANs/gan-vs-density-fitting.png)


- **(a)** GANs: generate samples that a classifier cannot tell from real ones
- **(b)** VAEs, flows, diffusion: fit a probability density to the real samples

UDL §15.1 lists what we want from such a model: *efficient sampling*,
*high-quality samples*, *coverage* of the whole training distribution, a
*well-behaved latent space* (every $\mathbf{z}$ decodes to a plausible
$\mathbf{x}$ and nearby $\mathbf{z}$ give similar $\mathbf{x}$), an
*interpretable* latent space, and, for probabilistic models, an *efficient
likelihood*. No family has all of them. GANs sample fast and sharply but can
drop modes; VAEs sample fast with good coverage but blurrily; diffusion models
(lecture 18) give the best samples but are slow. Module M2 has the full
comparison table.


## Outline

- Autoencoders and their limits
- Variational autoencoders: the intuition
- Deriving the VAE loss: the evidence lower bound (ELBO) and the reparameterization trick
- VAE examples, and where VAEs live today
- Generative adversarial networks: the minimax game, its failure modes, and adversarial losses as a tool
- Summary and the bridge to diffusion

# Autoencoders and their limits

An autoencoder compresses its input to a lower-dimensional latent code and then
reconstructs the input from that code. We start with what it does well, then
ask whether it can generate new data. It cannot, and the reason why motivates
the VAE.


## Dimensionality reduction with an autoencoder

![Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/autoencoder-diagram.png)


- $\mathbf{x} = \mathbf{d}(\mathbf{e}(\mathbf{x}))$ → **lossless**: nothing lost
  in reducing the dimension
- $\mathbf{x} \neq \mathbf{d}(\mathbf{e}(\mathbf{x}))$ → **lossy**: some
  information is lost and can't be recovered

Find the best encoder $\mathbf{e}$ and decoder $\mathbf{d}$:

$$
(e^*, d^*) = \underset{(e,d) \in E \times D}{\operatorname{argmin}} \; \epsilon\big(x, d(e(x))\big)
$$

where $\epsilon$ is the reconstruction error.


## PCA is the linear autoencoder

![Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/pca-projection.png)

- Project $n_d$-dimensional features onto an *orthogonal* $n_e$-dimensional
  subspace that minimizes Euclidean distance
- PC1 points in the direction of greatest variance

**A linear transformation.**


A one-linear-layer encoder and decoder (no activation) spans the same best
subspace, but need not converge to the PCA basis.

![Left: the best linear subspace for 3D data. Right: the PCA directions are orthogonal, but the AE can end up with any basis. Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/linear-ae-vs-pca.png)


## Neural network autoencoder

![Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/ae-dog-reconstruction.png)

$$
\text{loss} = \| x - \hat{x} \|^2 = \| x - d(z) \|^2 = \| x - d(e(x)) \|^2
$$

With nonlinear layers the encoder and decoder can fit a curved manifold, not
just a subspace. Example: `01_autoencoder_mnist.ipynb` (notebook: to be
re-hosted in the SP27 materials repo); CIFAR10 version in the same folder.


![](figs/17-VAEs-and-GANs/ae-mnist-example.png)


## Can we generate new samples with an autoencoder?

![Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/ae-generation-process.png)


- Train encoder and decoder as an autoencoder.
- Pick a *different* point in the latent space.
- Feed it to the decoder to generate an output.

**Will this produce a good quality output? Why?**


## Sampling and interpolating the latent space

Decode $z \sim \mathcal{N}(0, I)$:

![](figs/17-VAEs-and-GANs/ae-decode-standard-normal.png)

Decode $z \sim \mathcal{N}(\mu_{\text{train}}, \sigma^2_{\text{train}})$, a
Gaussian fitted to the training codes:

![](figs/17-VAEs-and-GANs/ae-decode-fitted-gaussian.png)


Linearly interpolate between two training codes:

$$
\mathrm{decode}\big((1-\alpha) z_A + \alpha z_B\big)
$$

![](figs/17-VAEs-and-GANs/ae-latent-interpolation.png)


Nothing in the training objective says what the decoder should do *between*
or *away from* the training codes. (`01_autoencoder_mnist.ipynb`)

## Extreme case: memorization

![Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/ae-memorization.png)

**The encoder and decoder are powerful enough to memorize the data.** The
latent space has no reason to be *regular*.

# Variational autoencoders: the intuition

A plain autoencoder gives no guarantee that points between or away from the
encoded training data decode to anything meaningful. A VAE fixes this by
encoding each input as a *distribution* over the latent space and regularizing
those distributions toward a standard normal. The name unpacks as:
**variational** inference, a method that approximates probability densities
through optimization; **autoencoder**, a network that learns codes for
unlabeled data; and **Bayes**, because the model is a joint density over
latent and observed variables.


## VAE: encode a distribution, not a point

> A VAE is an autoencoder whose encodings distribution is regularized during
> training to ensure that its latent space has good properties allowing us to
> generate new data. ([Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73))


![Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/vae-diagram.png)

- The encoder emits a mean vector $\mu_x$ and a vector $\sigma_x$ of standard
  deviations: independent Gaussians, one per latent dimension.
- Sample $z \sim \mathcal{N}(\mu_x, \sigma_x^2)$ (shorthand for the diagonal
  covariance $\mathrm{diag}(\sigma_x^2)$).
- Feed $z$ to the decoder to produce the output.


|  | encoding → | sampling → | decoding |
|:--|:--:|:--:|:--:|
| **autoencoder** | $\mathbf{z} = \mathbf{e}(\mathbf{x})$ | | $\mathbf{d}(\mathbf{z})$ |
| **VAE** | $p(\mathbf{z} \mid \mathbf{x})$ | $\mathbf{z} \sim p(\mathbf{z} \mid \mathbf{x})$ | $\mathbf{d}(\mathbf{z})$ |

## The VAE loss

$$
\begin{aligned}
\text{loss} &= \underbrace{\| x - \hat{x} \|^2}_{\text{L2 reconstruction}} + \underbrace{\mathrm{KL}\big[\mathcal{N}(\mu_x, \sigma_x^2) \,\|\, \mathcal{N}(0, I)\big]}_{\text{Kullback-Leibler divergence}} \\
&= \| x - d(z) \|^2 + \mathrm{KL}\big[\mathcal{N}(\mu_x, \sigma_x^2) \,\|\, \mathcal{N}(0, I)\big]
\end{aligned}
$$

The loss is the L2 loss of the autoencoder plus a KL-divergence term that pulls
every encoded distribution toward the standard normal.

**KL divergence** (UDL C.5.1): the most common measure of distance between
distributions $p(x)$ and $q(x)$,

$$
D_{KL}\big[p(x) \,\|\, q(x)\big] = \int p(x) \log\left[\frac{p(x)}{q(x)}\right] dx,
$$

non-negative, zero only when $p = q$, and not symmetric.


## Why regularize? A regular latent space

![Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/latent-space-regularity.png)

**Continuity**: nearby codes decode to similar outputs. **Completeness**: any
code drawn from the prior decodes to something meaningful.

## Encoding to Gaussians is not enough

![Left: without regularization. Right: with regularization. Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/normal-encodings-regularization.png)


![Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/regularization-gradient.png)


Left alone, the encoder shrinks every $\sigma_x$ to zero and spreads the
means apart, and we are back to the plain autoencoder. So we regularize the
means *and* the variances toward a standard normal:

$$
\text{loss} = \| x - \hat{x} \|^2 + \mathrm{KL}\big[\mathcal{N}(\mu_x, \sigma_x^2) \,\|\, \underline{\mathcal{N}(0, I)}\big]
$$

The continuity and completeness this buys create a "gradient" over the
information encoded in the latent space (right).

# Deriving the VAE loss

Now we make the intuition precise. Treating the latent code as a random
variable turns encoding into Bayesian inference. The posterior is intractable,
so we approximate it with variational inference, and the objective that falls
out, the evidence lower bound, is exactly the reconstruction-plus-KL loss
above, with the weighting between the two terms explained. This is the
derivation lecture 18 reuses for diffusion models, so it is worth following
every line. We follow Rocca's notation: $p(\cdot)$ for the model's densities
and $f, g, h$ for the decoder and the two encoder heads. In UDL's notation
(§17.3 to §17.5) the decoder is $\mathrm{Pr}(\mathbf{x} \mid \mathbf{z}, \boldsymbol{\phi})$
and the approximate posterior is $q(\mathbf{z} \mid \mathbf{x}, \boldsymbol{\theta})$.


## Preliminaries: a Bayesian latent-variable model

![](figs/17-VAEs-and-GANs/bayesian-graphical-model.png)

![](figs/17-VAEs-and-GANs/aevb-paper-header.png)


- $p(z)$: the **prior**, our belief about $z$ before seeing data
- $p(x \mid z)$: the **likelihood**, the probability of a sample given $z$

The joint density factorizes into the prior and the likelihood:

$$
p(\mathbf{z}, \mathbf{x}) = p(\mathbf{x} \mid \mathbf{z})\, p(\mathbf{z})
$$

Generating a sample means: draw $z$ from the prior, then draw $x$ from the
likelihood. That is the latent-variable pipeline from the introduction.

[Kingma and Welling, "Auto-Encoding Variational Bayes", 2013](https://arxiv.org/abs/1312.6114)


## Bayesian inference: the posterior

![Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/probabilistic-encoder-decoder.png)

- **Probabilistic encoder** $p(z \mid x)$: the *posterior*, our updated
  knowledge of $z$ given a sample
- **Probabilistic decoder** $p(x \mid z)$: the *likelihood*


$$
\underbrace{p(z \mid x)}_{\text{posterior}} = \frac{\overbrace{p(x \mid z)}^{\text{likelihood}}\;\overbrace{p(z)}^{\text{prior}}}{\underbrace{p(x)}_{\text{evidence}}}
$$

Bayes' theorem relates the posterior to the likelihood and the prior.


$$
= \frac{p(x \mid z)\, p(z)}{\int p(x \mid z)\, p(z)\, dz}
$$

The *evidence* $p(x)$ is an integral over every possible $z$. We can't compute
it, so we can't compute the posterior directly. We approximate it with
**variational inference**.


## Simplifying assumptions

Assume the *prior* is a standard Gaussian:

$$
p(z) \equiv \mathcal{N}(0, I)
$$

and the *likelihood* is a Gaussian centred on a decoder function $f$:

$$
p(x \mid z) \equiv \mathcal{N}(f(z), cI)
$$

where $f \in F$ is a family of functions we will specify later and $c > 0$.


Approximate the *posterior* $p(z \mid x)$ by a Gaussian $q_x(z)$ with a
diagonal covariance:

$$
q_x(z) \equiv \mathcal{N}\big(g(x), \mathrm{diag}(h(x)^2)\big)
$$

where $g \in G$ gives the mean $\mu_x = g(x)$ and $h \in H$ gives the vector
of standard deviations $\sigma_x = h(x)$, so the covariance is
$\mathrm{diag}(\sigma_x^2)$.


Defining $h$ as the standard deviation rather than the covariance keeps the
reparameterization $z = \sigma_x \zeta + \mu_x$ below exact; if $h$ were the
covariance we would need $\sigma_x = \sqrt{h(x)}$ there.


## Variational inference: the objective

$$
(g^*, h^*) = \underset{(g,h) \in G \times H}{\arg\min}\; KL\big(q_x(z) \,\|\, p(z \mid x)\big)
$$

Find the encoder functions $g$ and $h$ that make $q_x(z)$ closest, in KL
divergence, to the true posterior $p(z \mid x)$.

The catch: $p(z \mid x)$ contains the evidence $p(x)$ that we cannot compute.
The next slide shows that it drops out.


## Deriving the ELBO

Starting from the KL divergence between the approximate and the true posterior:
line 2 writes the KL as an expectation, splits the log of a quotient, and
substitutes Bayes' theorem for the posterior; line 3 splits the log of a
product into a sum of logs; line 4 is the step the usual presentation skips:
$p(x)$ does not depend on $z$, so $\mathbb{E}_{z \sim q_x}[\log p(x)] = \log p(x)$,
and it does not depend on the encoder functions $(g, h)$ either, so it is a
constant in this $\arg\min$ and can be dropped; line 5 negates (argmin to
argmax) and collects the remaining $q_x$ and $p(z)$ terms into a KL divergence.


$$
(g^*, h^*) = \underset{(g,h)}{\arg\min}\; KL\big(q_x(z) \,\|\, p(z \mid x)\big)
$$

$$
= \underset{(g,h)}{\arg\min}\; \left(\mathbb{E}_{z \sim q_x}\big[\log q_x(z)\big] - \mathbb{E}_{z \sim q_x}\left[\log \frac{p(x \mid z)\, p(z)}{p(x)}\right]\right)
$$


$$
= \underset{(g,h)}{\arg\min}\; \Big(\mathbb{E}_{z \sim q_x}[\log q_x(z)] - \mathbb{E}_{z \sim q_x}[\log p(z)] - \mathbb{E}_{z \sim q_x}[\log p(x \mid z)] + \mathbb{E}_{z \sim q_x}[\log p(x)]\Big)
$$


$$
= \underset{(g,h)}{\arg\min}\; \Big(\mathbb{E}_{z \sim q_x}[\log q_x(z)] - \mathbb{E}_{z \sim q_x}[\log p(z)] - \mathbb{E}_{z \sim q_x}[\log p(x \mid z)]\Big) + \underbrace{\log p(x)}_{\text{constant in } (g,h)}
$$


$$
= \underset{(g,h)}{\arg\max}\; \Big(\underbrace{\mathbb{E}_{z \sim q_x}[\log p(x \mid z)]}_{\substack{\text{maximize the expected}\\ \text{log likelihood}}} - \underbrace{KL\big(q_x(z) \,\|\, p(z)\big)}_{\substack{\text{keep the approximate posterior}\\ \text{close to the prior}}}\Big)
$$



## The evidence lower bound (ELBO)

Rearranging line 3 of the derivation (no approximation yet):

$$
\log p(x) = \underbrace{\mathbb{E}_{z \sim q_x}[\log p(x \mid z)] - KL\big(q_x(z) \,\|\, p(z)\big)}_{\text{ELBO}} + \underbrace{KL\big(q_x(z) \,\|\, p(z \mid x)\big)}_{\geq 0}
$$

- The last KL is $\geq 0$, so $\text{ELBO} \leq \log p(x)$: the bracket is a
  **lower bound on the log evidence**, hence the name.
- For fixed decoder $f$, $\log p(x)$ is a constant, so *maximizing the ELBO over
  the encoder* $(g, h)$ is the same as *minimizing the gap* $KL(q_x \,\|\, p(z \mid x))$:
  the variational objective we started with.
- $\log p(x) = \int p(x \mid z)\,p(z)\,dz$ *does* depend on the decoder $f$. We
  cannot maximize it directly, so we maximize its lower bound over $f$ too.
  Raising a lower bound raises the thing it bounds (or tightens the bound).


UDL §17.3 arrives at the same inequality through Jensen's inequality
(§17.3) and writes it as
$\log \mathrm{Pr}(\mathbf{x} \mid \boldsymbol{\phi}) \geq \text{ELBO}[\boldsymbol{\theta}, \boldsymbol{\phi}]$.
The two routes give the same bound; this one makes the size of the gap
explicit. The ELBO is the quantity to remember for lecture 18: a diffusion
model's loss is the ELBO of a latent-variable model with many latents and a
fixed encoder.

**Check the identity numerically.** Take a one-dimensional model small enough
that everything is exact: prior $p(z) = \mathcal{N}(0, 1)$, a linear decoder
$f(z) = wz + b$ and likelihood $p(x \mid z) = \mathcal{N}(f(z), c)$. Then
$p(x) = \mathcal{N}(b, w^2 + c)$, the true posterior $p(z \mid x)$ is Gaussian,
and for $q_x = \mathcal{N}(\mu_x, \sigma_x^2)$ the reconstruction term has the
closed form
$\mathbb{E}_{q_x}[\log p(x \mid z)] = \log \mathcal{N}(x;\, w\mu_x + b,\, c) - w^2\sigma_x^2 / 2c$.
For every choice of $q_x$ the ELBO and the gap add up to the same $\log p(x)$,
and the gap closes only when $q_x$ is the true posterior.

In [ ]:
#| code-fold: true
#| fig-cap: "ELBO as a function of the encoder mean $\\mu_x$ for three encoder standard deviations $\\sigma_x$, in the 1D linear-Gaussian model with $w = 2$, $b = 0.5$, $c = 0.5$, $x = 3$. Every curve stays below $\\log p(x)$ and touches it only at the true posterior."
#| fig-alt: "Three concave curves of the ELBO against mu_x from -0.5 to 2.5, below a dashed horizontal line at log p(x) = -2.37. The curve for sigma_x = 0.333 (the true posterior standard deviation) touches the line at mu_x = 1.11; the curves for sigma_x = 0.2 and 0.6 peak at the same mu_x but stay below it."
import numpy as np
import matplotlib.pyplot as plt

w, b, c = 2.0, 0.5, 0.5     # decoder f(z) = w z + b, likelihood variance c
x = 3.0                     # the observed data point

def log_normal(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - (x - mean) ** 2 / (2 * var)

def kl_gauss(m1, s1, m2, s2):   # KL(N(m1, s1^2) || N(m2, s2^2))
    return np.log(s2 / s1) + (s1**2 + (m1 - m2) ** 2) / (2 * s2**2) - 0.5

def elbo(mu_x, sigma_x):
    recon = log_normal(x, w * mu_x + b, c) - w**2 * sigma_x**2 / (2 * c)
    return recon - kl_gauss(mu_x, sigma_x, 0.0, 1.0)

log_px = log_normal(x, b, w**2 + c)                 # evidence: integral done by hand
post_var = 1 / (1 + w**2 / c)                       # true posterior p(z | x)
post_mu, post_sd = post_var * w * (x - b) / c, np.sqrt(post_var)

print(f"log p(x) = {log_px:.4f};  true posterior N({post_mu:.3f}, {post_sd:.3f}^2)")
print(f"{'q_x':>18}{'ELBO':>10}{'gap KL(q||p(z|x))':>20}{'ELBO + gap':>12}")
for m, s in [(0.0, 1.0), (1.0, 0.5), (post_mu, 0.6), (post_mu, post_sd)]:
    e, gap = elbo(m, s), kl_gauss(m, s, post_mu, post_sd)
    print(f"{f'N({m:.3f}, {s:.3f}^2)':>18}{e:>10.4f}{gap:>20.4f}{e + gap:>12.4f}")

mus = np.linspace(-0.5, 2.5, 301)
fig, ax = plt.subplots(figsize=(7, 3.5))
for s in [0.2, post_sd, 0.6]:
    ax.plot(mus, elbo(mus, s), label=rf"ELBO, $\sigma_x = {s:.3f}$")
ax.axhline(log_px, color="k", ls="--", label=r"$\log p(x)$")
ax.set_ylim(log_px - 4, log_px + 0.5)
ax.set_xlabel(r"encoder mean $\mu_x$")
ax.set_ylabel("nats")
ax.legend(loc="lower center")
plt.show()

## Substituting the Gaussian likelihood

With $p(x \mid z) \equiv \mathcal{N}(f(z), cI)$, the log likelihood is a scaled
squared error plus a constant that does not depend on $f$, $g$ or $h$:

$$
\log p(x \mid z) = -\frac{\|x - f(z)\|^2}{2c} + \text{const}
$$

so the objective becomes

$$
(f^*, g^*, h^*) = \underset{(f,g,h) \in F \times G \times H}{\arg\max}\; \left(\mathbb{E}_{z \sim q_x}\left[-\frac{\|x - f(z)\|^2}{2c}\right] - KL\big(q_x(z) \,\|\, p(z)\big)\right)
$$

This brings the decoder $f$ into the equation: we now optimize the decoder and
both encoder heads together.


The constant $c$, the variance we assumed for the likelihood, sets the balance
between reconstruction error and the KL regularizer. A small $c$ (we trust the
decoder to reconstruct exactly) weights reconstruction heavily; a large $c$
weights the regularizer. This is the $\beta$ of a $\beta$-VAE in disguise.


## The KL term in closed form

Both $q_x(z)$ and $p(z)$ are Gaussians, so the KL term needs no sampling. In
one dimension, with $q = \mathcal{N}(\mu, \sigma^2)$ and $p = \mathcal{N}(0, 1)$:

$$
\begin{aligned}
KL(q \,\|\, p) &= \mathbb{E}_{z \sim q}\big[\log q(z) - \log p(z)\big] \\
&= \mathbb{E}_{z \sim q}\left[-\tfrac{1}{2}\log(2\pi\sigma^2) - \frac{(z-\mu)^2}{2\sigma^2} + \tfrac{1}{2}\log(2\pi) + \frac{z^2}{2}\right] \\
&= -\tfrac{1}{2}\log\sigma^2 - \tfrac{1}{2} + \tfrac{1}{2}\big(\sigma^2 + \mu^2\big)
\end{aligned}
$$

using $\mathbb{E}_q[(z-\mu)^2] = \sigma^2$ and $\mathbb{E}_q[z^2] = \sigma^2 + \mu^2$.

The $D$ latent dimensions are independent, so the KL divergences add:

$$
KL\Big[\mathcal{N}\big(\mu_x, \mathrm{diag}(\sigma_x^2)\big) \,\Big\|\, \mathcal{N}(0, I)\Big] = \frac{1}{2}\sum_{d=1}^{D}\Big(\sigma_{x,d}^2 + \mu_{x,d}^2 - 1 - \log\sigma_{x,d}^2\Big)
$$

Each term is zero exactly when $\mu_{x,d} = 0$ and $\sigma_{x,d} = 1$. The
reconstruction expectation, by contrast, has no closed form: we estimate it by
sampling.


**Check the formula.** The closed form against a Monte-Carlo estimate of the
definition $\mathbb{E}_{z \sim q}[\log q(z) - \log p(z)]$ for a 3-dimensional
code. Change `mu` and `sigma`: the third dimension, at $\mu = 0$, $\sigma = 1$,
contributes nothing, and a small $\sigma$ costs more than a shifted $\mu$.

In [ ]:
import numpy as np

mu = np.array([0.5, -1.0, 0.0])      # encoder mean mu_x = g(x)
sigma = np.array([0.8, 0.3, 1.0])    # encoder standard deviations sigma_x = h(x)
n_samples = 200_000

kl_per_dim = 0.5 * (sigma**2 + mu**2 - 1 - np.log(sigma**2))

rng = np.random.default_rng(0)
z = mu + sigma * rng.standard_normal((n_samples, mu.size))           # z ~ q_x
log_q = np.sum(-0.5 * np.log(2 * np.pi * sigma**2) - (z - mu) ** 2 / (2 * sigma**2), axis=1)
log_p = np.sum(-0.5 * np.log(2 * np.pi) - z**2 / 2, axis=1)          # N(0, I)
kl_mc = np.mean(log_q - log_p)
kl_se = np.std(log_q - log_p) / np.sqrt(n_samples)

print(f"closed form  : {kl_per_dim.sum():.4f} nats  (per dimension {np.round(kl_per_dim, 4)})")
print(f"Monte Carlo  : {kl_mc:.4f} +/- {kl_se:.4f} nats from {n_samples:,} samples")

## Enter the neural networks

![](figs/17-VAEs-and-GANs/vae-encoder-decoder-networks.png)

The encoder produces the mean $\mu_x = g(x)$ and the standard deviations
$\sigma_x = h(x)$ from a shared trunk with two heads.


The decoder $f$ reconstructs the input during training and generates from
$z \sim \mathcal{N}(0, I)$ afterwards.


## One more problem: sampling blocks backprop


![We can't backpropagate through a sampling step. Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/sampling-blocks-backprop.png)

![The reparameterization trick. Left: sampling prevents backpropagation and therefore training. Right: with $\zeta \sim \mathcal{N}(0, I)$ and $z = \sigma_x \zeta + \mu_x$, the randomness is an *input* and gradients flow to $\mu_x$ and $\sigma_x$. Source: [Rocca, 2019](https://medium.com/data-science/understanding-variational-autoencoders-vaes-f70510919f73).](figs/17-VAEs-and-GANs/reparameterization-trick.png)


**The reparameterization trick:** sample $\zeta \sim \mathcal{N}(0, I)$ and set
$z = \sigma_x \odot \zeta + \mu_x$. Then $z \sim \mathcal{N}(\mu_x, \sigma_x^2)$
exactly as before, but the randomness enters as an *input* and $z$ is a
deterministic, differentiable function of $\mu_x$ and $\sigma_x$.

## What happens during backprop

**$\zeta$ is sampled during the forward pass and held fixed during backprop
for that training step.**

1. **Forward pass:**
   - Input $x$ goes through the encoder → $\mu_x(x; \phi)$ and
     $\sigma_x(x; \phi)$, both deterministic functions of the encoder weights $\phi$
   - Draw $\zeta \sim \mathcal{N}(0, I)$: a random vector **independent of all weights**
   - Compute $z = \sigma_x \odot \zeta + \mu_x$
   - Feed $z$ through the decoder → reconstruction → loss $\mathcal{L}$


2. **Backward pass:**
   - $\zeta$ is a **constant**, like an input to the network. Once sampled it is
     just a fixed number for this step.
   - $z$ depends on the weights through *two* paths:
     $$
     \frac{\partial z}{\partial \mu_x} = 1, \qquad \frac{\partial z}{\partial \sigma_x} = \zeta
     $$
   - so the chain rule for the reconstruction term gives
     $$
     \frac{\partial \mathcal{L}}{\partial \phi} = \frac{\partial \mathcal{L}}{\partial z}\left(\frac{\partial \mu_x}{\partial \phi} + \zeta \odot \frac{\partial \sigma_x}{\partial \phi}\right) + \frac{\partial \mathcal{L}_{KL}}{\partial \phi}
     $$
   The $\zeta$ in the gradient is the **same fixed number** drawn in the forward
   pass. The KL term depends on $\mu_x$ and $\sigma_x$ directly, with no sampling.


Without the trick we would need a gradient of an *expectation over a
distribution whose parameters we are learning*; with it, the expectation is over
a fixed distribution ($\zeta$) and the parameters sit inside a deterministic
function, so an ordinary Monte-Carlo estimate (usually a single draw) is
differentiable. The same device reappears in lecture 18: every diffusion step
in UDL's $\mathbf{z}_t$ notation (written $\mathbf{x}_t$ in lecture 18),
$\mathbf{z}_t = \sqrt{1-\beta_t}\,\mathbf{z}_{t-1} + \sqrt{\beta_t}\,\boldsymbol{\epsilon}$,
is a reparameterized sample.

**Why not just differentiate the density?** There is another unbiased estimator
that needs no reparameterization, the *score-function* (REINFORCE) estimator
$\nabla \mathbb{E}_{q}[F(z)] = \mathbb{E}_{q}\big[F(z)\, \nabla \log q(z)\big]$
(UDL problem 17.5). Compare the two on a 1D example with a known answer:
$F(z) = (z - a)^2$ with $z \sim \mathcal{N}(\mu, \sigma^2)$, so
$\mathbb{E}[F] = (\mu - a)^2 + \sigma^2$, $\partial/\partial\mu = 2(\mu - a)$ and
$\partial/\partial\sigma = 2\sigma$. The reparameterized estimator uses
$\partial F/\partial z$ at $z = \mu + \sigma\zeta$, times
$\partial z/\partial\mu = 1$ and $\partial z/\partial\sigma = \zeta$. Both are
right on average; the reparameterized one has several times less spread, and
that is why VAEs train with one sample per step.

In [ ]:
#| code-fold: true
#| fig-cap: "2000 estimates of $\\partial \\mathbb{E}[(z-a)^2] / \\partial\\sigma$, each from 100 samples, with $\\mu = 1$, $\\sigma = 0.5$, $a = 2$. Both estimators are centred on the true value 1; the reparameterized one is far tighter."
#| fig-alt: "Two overlaid histograms of gradient estimates with a vertical line at the true value 1. The reparameterization histogram is narrow, roughly from 0.3 to 1.8; the score-function histogram is about four times wider, roughly from -1 to 4."
# Adapted from UDL notebook 17.2 (MIT)
import numpy as np
import matplotlib.pyplot as plt

mu, sigma = 1.0, 0.5        # parameters of q = N(mu, sigma^2)
a = 2.0                     # F(z) = (z - a)^2
n_samples = 100             # samples per gradient estimate
n_repeats = 2000            # independent estimates, to measure the spread

rng = np.random.default_rng(0)
zeta = rng.standard_normal((n_repeats, n_samples))
z = mu + sigma * zeta                       # reparameterized samples
F = (z - a) ** 2
dF_dz = 2 * (z - a)

reparam = {"mu": np.mean(dF_dz * 1, axis=1),          # dz/dmu = 1
           "sigma": np.mean(dF_dz * zeta, axis=1)}    # dz/dsigma = zeta
score = {"mu": np.mean(F * (z - mu) / sigma**2, axis=1),                      # d log q / d mu
         "sigma": np.mean(F * ((z - mu) ** 2 - sigma**2) / sigma**3, axis=1)}  # d log q / d sigma
analytic = {"mu": 2 * (mu - a), "sigma": 2 * sigma}

print(f"{'':16}{'d/d mu':>20}{'d/d sigma':>20}")
print(f"{'analytic':16}{analytic['mu']:>20.3f}{analytic['sigma']:>20.3f}")
for name, est in [("reparameterized", reparam), ("score function", score)]:
    cols = "".join(f"{est[k].mean():>10.3f} sd {est[k].std():5.3f}" for k in ("mu", "sigma"))
    print(f"{name:16}{cols}")

fig, ax = plt.subplots(figsize=(7, 3))
bins = np.linspace(-1.5, 4.5, 61)
ax.hist(score["sigma"], bins=bins, alpha=0.6, label="score function")
ax.hist(reparam["sigma"], bins=bins, alpha=0.6, label="reparameterized")
ax.axvline(analytic["sigma"], color="k", ls="--", label="true value")
ax.set_xlabel(r"estimate of $\partial\,\mathbb{E}[(z-a)^2] / \partial\sigma$")
ax.set_ylabel("count")
ax.legend()
plt.show()

## Putting it all together

![](figs/17-VAEs-and-GANs/vae-full-architecture.png)


- Replace the expectation of the reconstruction term by a single-draw
  Monte-Carlo estimate
- Write $C = 1/(2c)$


$$
\begin{aligned}
\text{loss} &= C \| x - \hat{x} \|^2 + \mathrm{KL}\big[\mathcal{N}(\mu_x, \sigma_x^2) \,\|\, \mathcal{N}(0, I)\big] \\
&= C \| x - f(z) \|^2 + \mathrm{KL}\big[\mathcal{N}\big(g(x), \mathrm{diag}(h(x)^2)\big) \,\|\, \mathcal{N}(0, I)\big]
\end{aligned}
$$

**We have recovered the intuitive loss, and we have a trainable neural network.**


## The VAE training loop

```python
for x in loader:                                 # x: (B, D_x)
    mu, log_var = encoder(x)                     # two heads, each (B, D_z)
    zeta = torch.randn_like(mu)                  # zeta ~ N(0, I), no grad needed
    z = mu + torch.exp(0.5 * log_var) * zeta     # reparameterization: sigma = exp(log_var / 2)
    x_hat = decoder(z)

    recon = ((x - x_hat) ** 2).sum(dim=1)        # ||x - f(z)||^2
    kl = 0.5 * (log_var.exp() + mu**2 - 1 - log_var).sum(dim=1)   # closed-form KL
    loss = (C * recon + kl).mean()

    opt.zero_grad(); loss.backward(); opt.step()
```

- The encoder predicts $\log \sigma_x^2$ rather than $\sigma_x$, so it is
  unconstrained and the KL formula needs no square root.
- One $\zeta$ per example per step is the usual Monte-Carlo estimate.
- To **generate**: `decoder(torch.randn(n, D_z))`. The encoder is not used.

Notebooks: `03_vae.ipynb`, `04_vae_mnist.ipynb`, `05_vae_cifar10.ipynb`
(notebook: to be re-hosted in the SP27 materials repo).

# VAE examples, and where VAEs live today

With a trained VAE we can sample new data, edit real data by moving its code
through the latent space, and use the learned latent structure for tasks such
as debiasing and outlier detection. The last slide of this section is the one
that matters most for the rest of the unit: the latent space that modern
diffusion models run in is a VAE's.


## Generating high-quality images

![Vahdat & Kautz (2020), ["NVAE: A deep hierarchical variational autoencoder"](https://arxiv.org/abs/2007.03898).](figs/17-VAEs-and-GANs/nvae-faces.png)

A plain VAE produces blurry samples (the decoder averages over everything a
code could mean). Hierarchies of latents at several resolutions, as in NVAE,
close much of the gap.

## Resynthesizing real data with changes

![Resynthesis. The original image on the left is projected into the latent space using the encoder, and the mean of the predicted Gaussian represents the image. The center-left image in the grid is the reconstruction. The other images are reconstructions after moving the code in directions representing smiling/neutral (horizontal) and mouth open/closed (vertical). UDL Fig. 17.13, adapted from White (2016).](figs/17-VAEs-and-GANs/vae-resynthesis.png)

## Disentanglement of the latent space

![UDL Fig. 17.14, adapted from Chen, Li, Grosse & Duvenaud (2018), [*Isolating sources of disentanglement in VAEs*](https://arxiv.org/abs/1802.04942), NeurIPS ($\beta$-TCVAE).](figs/17-VAEs-and-GANs/disentanglement-chairs.png)

A *disentangled* latent space has one interpretable factor per dimension.
Weighting the KL term more heavily ($\beta$-VAE) encourages it, at a cost in
reconstruction.

## Learned data manifolds

![Visualisations of learned data manifolds for generative models with a two-dimensional latent space, learned with AEVB. Since the prior is Gaussian, linearly spaced coordinates on the unit square were transformed through the inverse CDF of the Gaussian to produce values of $\mathbf{z}$; for each, the mean of $p_{\boldsymbol{\theta}}(\mathbf{x} \mid \mathbf{z})$ is plotted. Figure 4 of [Kingma and Welling, 2013](https://arxiv.org/abs/1312.6114).](figs/17-VAEs-and-GANs/aevb-learned-manifolds.png)

## Conditional VAEs

Feed a label or attribute vector $\mathbf{c}$ to both the encoder and the
decoder: $q_x(z \mid \mathbf{c})$ and $p(x \mid z, \mathbf{c})$. The latent
space then only has to explain what the condition does not.

Example from [Kana, 2020](https://medium.com/data-science/variational-autoencoders-vaes-for-dummies-step-by-step-tutorial-69e6d1c9d8e9),
trained on CelebA attributes.


![](figs/17-VAEs-and-GANs/conditional-vae-faces.png)


## Using the latent structure: debiasing and outlier detection

![© Alexander Amini and Ava Amini, MIT 6.S191: Introduction to Deep Learning, IntroToDeepLearning.com](figs/17-VAEs-and-GANs/debiasing-faces.png)

**Debiasing.** A VAE trained alongside a face classifier learns the latent
structure of the dataset; the learned latent densities then re-weight rare
combinations (skin tone, pose, lighting) upward during training.
[Amini et al., 2019](https://dl.acm.org/doi/10.1145/3306618.3314243).

![Debiasing VAE: the unsupervised latent variables adaptively resample the dataset while training the classifier. Figure 2 of Amini et al., 2019.](figs/17-VAEs-and-GANs/db-vae-architecture.png)


![© Alexander Amini and Ava Amini, MIT 6.S191: Introduction to Deep Learning, IntroToDeepLearning.com](figs/17-VAEs-and-GANs/outlier-detection-driving.png)

**Outlier detection.** For end-to-end driving, the same encoder predicts the
steering command *and* unsupervised latents describing the scene; a low
likelihood under the learned latent density flags a situation the model was not
trained for.
[Amini et al., 2018](https://ieeexplore.ieee.org/abstract/document/8594386).

![Semi-supervised end-to-end control: an encoder learns a supervised control command and unsupervised latents that describe the image, enabling novelty detection and dataset debiasing. Figure 1 of Amini et al., 2018.](figs/17-VAEs-and-GANs/novelty-detection-architecture.png)


## Where VAEs are used

- **Molecules and biology**
  - Drug discovery: a continuous latent space over SMILES strings to interpolate and optimize
    ([Gómez-Bombarelli et al., 2018](https://pubs.acs.org/doi/full/10.1021/acscentsci.7b00572))
  - Single-cell RNA-seq: scVI models expression counts and batch effects
    ([Lopez et al., 2018](https://www.nature.com/articles/s41592-018-0229-2))
  - Protein sequence generation ([Greener et al., 2018](https://www.nature.com/articles/s41598-018-34533-1))
- **Social science and text**
  - Neural topic models: ProdLDA ([Srivastava & Sutton, 2017](https://arxiv.org/abs/1703.01488))
  - Imputing heterogeneous, incomplete survey data: HI-VAE ([Nazábal et al., 2020](https://hdl.handle.net/10016/32743))
  - Synthetic, privacy-preserving health data ([Pfitzner & Arnrich, 2022](https://arxiv.org/abs/2211.11591))
- **Business**
  - Anomaly and fraud detection by reconstruction probability ([An & Cho, 2015](http://dm.snu.ac.kr/static/docs/TR/SNUDM-TR-2015-03.pdf))
  - Collaborative filtering: Mult-VAE ([Liang et al., 2018](https://arxiv.org/abs/1802.05814))
  - Customer segmentation by clustering in the latent space: VaDE ([Jiang et al., 2017](https://arxiv.org/abs/1611.05148))


![Gómez-Bombarelli et al., ACS Central Science, 2018](figs/17-VAEs-and-GANs/molecule-vae.png)

*Survey by Claude.* The common thread: the task needs a *latent space* (to
interpolate, cluster, or score likelihood), not just samples.



## VAEs in 2026: the latent space of diffusion models

Stable Diffusion, FLUX, and most image and video generators do not diffuse
pixels. They diffuse in the latent space of a **VAE**:

- The autoencoder is trained first, with a tiny KL weight (so it is closer to a
  regularized autoencoder than to the VAE above), a perceptual loss, and a
  **patch-based adversarial loss**, the recipe from VQGAN
  ([Esser et al., 2021](https://arxiv.org/abs/2012.09841)).
- Stable Diffusion's encoder downsamples $512 \times 512 \times 3$ pixels by a
  factor of 8 to a $64 \times 64 \times 4$ latent; the diffusion model never
  sees a pixel ([Rombach et al., 2022](https://arxiv.org/abs/2112.10752)).
- Later models widen the latent to 16 channels: reconstruction quality of the
  VAE bounds the quality of everything generated on top of it
  ([Esser et al., 2024](https://arxiv.org/abs/2403.03206)).


![](figs/17-VAEs-and-GANs/latent-variable-architectures.png)

<small>© Alexander Amini and Ava Amini, MIT 6.S191</small>

So the two halves of this lecture meet inside one component: a VAE encoder and
decoder, trained with a GAN discriminator. That is our cue to look at GANs.


# Generative adversarial networks

Of the model families in the taxonomy, GANs are latent-variable models that are
generative but not probabilistic: they learn to generate samples without ever
writing down $\mathrm{Pr}(\mathbf{x})$. Instead of a likelihood, the training
signal is a second network that tries to tell generated samples from real ones.
This section gives the minimax game, the one derivation worth knowing (what the
optimal discriminator computes), why GANs are hard to train, and the two
conditional variants that made image translation possible. Module M2 has the
architectures (DCGAN, progressive growing, StyleGAN) and the tricks.


## The idea: a generator and a discriminator

![Figure © Alexander Amini and Ava Amini, MIT 6.S191: Introduction to Deep Learning, [IntroToDeepLearning.com](http://introtodeeplearning.com).](figs/17-VAEs-and-GANs/gan-generator-discriminator.png)

- Don't build a probability model at all. Learn a transformation
  $\mathbf{x}^* = \mathbf{g}[\mathbf{z}, \boldsymbol{\theta}]$ from noise
  $\mathbf{z}$ to samples that look like the data.
- The **generator** turns noise into an imitation of the data to try to fool the
  discriminator.
- The **discriminator** $\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]$ is a
  binary classifier that tries to tell real data from the generator's fakes.
- Train them *against each other*; when the discriminator can do no better than
  chance, the generated distribution matches the data.

[Goodfellow et al., "Generative Adversarial Nets", 2014](https://arxiv.org/abs/1406.2661)

## A one-dimensional example

$$
x_j^* = \mathrm{g}[z_j, \theta] = z_j + \theta
$$

![(a) $\theta = 3.0$, (b) $\theta = 4.9$, (c) $\theta = 6.7$. UDL ch. 15.](figs/17-VAEs-and-GANs/gan-example-step-c.png)

- <span style="color:#5fb3a8">Real</span> samples from a shifted Gaussian;
  <span style="color:#c97b5a">synthesized</span> samples $z_j$ from a standard
  Gaussian, shifted by $\theta$
- Train the discriminator $\mathrm{f}[\cdot, \phi]$, a logistic regression,
  to separate them: real if $\mathrm{sig}[\mathrm{f}[\cdot]] \geq 0.5$


- Update $\theta$ to *increase* the discriminator's loss (move the samples
  right); retrain the discriminator to *decrease* it
- Repeat until the discriminator does no better than chance (c)
- [UDL notebook 15.1](https://github.com/udlbook/udlbook/blob/main/Notebooks/Chap15/15_1_GAN_Toy_Example.ipynb)


## The discriminator loss

The **discriminator** is a binary classifier trained with cross-entropy
(lecture 3, binary classification loss), with $y = 1$ for real and $y = 0$ for
generated:

$$
\hat{\boldsymbol{\phi}} = \underset{\boldsymbol{\phi}}{\mathrm{argmin}} \left[ \sum_{i} -(1-y_i) \log\Bigl[1 - \mathrm{sig}[\mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}]]\Bigr] - y_i \log\Bigl[\mathrm{sig}[\mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}]]\Bigr] \right]
$$

Split the sum into generated samples $\mathbf{x}_j^*$ (where $y_j = 0$) and
real examples $\mathbf{x}_i$ (where $y_i = 1$):

$$
\hat{\boldsymbol{\phi}} = \underset{\boldsymbol{\phi}}{\mathrm{argmin}} \left[ \sum_{j} -\log\Bigl[1 - \mathrm{sig}[\mathrm{f}[\mathbf{x}_j^*, \boldsymbol{\phi}]]\Bigr] - \sum_{i} \log\Bigl[\mathrm{sig}[\mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}]]\Bigr] \right]
$$

- First sum: generated samples should get **low** probability of being real
- Second sum: real examples should get **high** probability of being real


## The minimax game

Substitute $\mathbf{x}_j^* = \mathbf{g}[\mathbf{z}_j, \boldsymbol{\theta}]$.
The **generator** adjusts $\boldsymbol{\theta}$ to make generated samples more
likely under the discriminator, i.e. to make the discriminator's loss *larger*:

$$
\hat{\boldsymbol{\phi}}, \hat{\boldsymbol{\theta}} = \underset{\boldsymbol{\theta}}{\mathrm{argmax}} \left[ \underset{\boldsymbol{\phi}}{\mathrm{argmin}} \left[ \sum_{j} -\log\Bigl[1 - \mathrm{sig}[\mathrm{f}[\mathbf{g}[\mathbf{z}_j, \boldsymbol{\theta}], \boldsymbol{\phi}]]\Bigr] - \sum_{i} \log\Bigl[\mathrm{sig}[\mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}]]\Bigr] \right] \right]
$$

![UDL ch. 15.](figs/17-VAEs-and-GANs/gan-loss-diagram-3.png)


Two losses, one for each player:

$$
\begin{aligned}
L[\boldsymbol{\phi}] = &\sum_{j} -\log\Bigl[1 - \mathrm{sig}[\mathrm{f}[\mathbf{x}_j^*, \boldsymbol{\phi}]]\Bigr] \\
&- \sum_{i} \log\Bigl[\mathrm{sig}[\mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}]]\Bigr]
\end{aligned}
$$

$$
L[\boldsymbol{\theta}] = \sum_{j} \log\Bigl[1 - \mathrm{sig}[\mathrm{f}[\mathbf{g}[\mathbf{z}_j, \boldsymbol{\theta}], \boldsymbol{\phi}]]\Bigr]
$$

The real-example sum does not depend on $\boldsymbol{\theta}$, so it drops
out of the generator's loss.


## Training: alternate the two players

```python
for c_gan_iter in range(n_gan_iters):
    # Run generator to produce synthesized data
    x_syn = generator(z, theta)

    # Update/train the discriminator (minimize L[phi])
    phi = update_discriminator(x_real, x_syn, n_iter_discrim, phi)

    # Update/train the generator (minimize L[theta])
    theta = update_generator(z, theta, n_iter_gen, phi)
```

After training, discard the discriminator; generate by drawing a new
$\mathbf{z}$ and passing it through the generator.


- The solution is a **Nash equilibrium**: neither player gains by changing only
  its own parameters.
- It sits at a **saddle point** of the shared loss (a minimum in
  $\boldsymbol{\phi}$, a maximum in $\boldsymbol{\theta}$), not at a minimum.
- Alternating gradient steps need not converge to a saddle: they can orbit it,
  oscillate, or diverge. GAN training is inherently unstable.


## What the optimal discriminator computes

Goodfellow et al. (2014), Section 4.1. Write $D(\mathbf{x}) = \mathrm{sig}[\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]]$
and replace the sums over samples by expectations over the real density
$p_{\text{data}}$ and the generated density $p_g$.


For a fixed generator, the discriminator maximizes (the negative of its loss,
in the population limit)

$$
V(D) = \int \Big[\, p_{\text{data}}(\mathbf{x}) \log D(\mathbf{x}) + p_g(\mathbf{x}) \log\big(1 - D(\mathbf{x})\big) \Big]\, d\mathbf{x}
$$

It can choose $D(\mathbf{x})$ independently at every $\mathbf{x}$, and
$a \log y + b \log(1 - y)$ is maximized at $y = a / (a + b)$, so

$$
D^*(\mathbf{x}) = \frac{p_{\text{data}}(\mathbf{x})}{p_{\text{data}}(\mathbf{x}) + p_g(\mathbf{x})}
$$

The optimal discriminator is a **density ratio**: it outputs $\tfrac{1}{2}$
wherever the two densities agree.


Substituting $D^*$ back, the generator's objective becomes

$$
C(G) = -\log 4 + 2\, D_{JS}\big[p_{\text{data}} \,\|\, p_g\big]
$$

so with an optimal discriminator, **training the generator minimizes the
Jensen-Shannon divergence** between the data and generated distributions,
with minimum $-\log 4$ exactly when $p_g = p_{\text{data}}$.


The last step uses $\log(p_{\text{data}} / (p_{\text{data}} + p_g)) = -\log 2 + \log(p_{\text{data}} / M)$
with $M = (p_{\text{data}} + p_g)/2$ on each term, which turns the two
expectations into the two KL divergences of the Jensen-Shannon divergence
defined on the Wasserstein slide below; the two $-\log 2$ terms give the
$-\log 4$ in $C(G)$. UDL §15.2.1 does the same calculation with sums.

**Check it numerically.** Take $p_{\text{data}} = \mathcal{N}(0, 1)$ and a
generator that outputs $\mathcal{N}(\theta, 1)$, as in the one-dimensional
example, and do the integrals on a fine grid. $V(D^*)$ matches
$-\log 4 + 2 D_{JS}$, no other discriminator does better, and $C(G)$ falls to
$-\log 4 \approx -1.386$ as $\theta \to 0$.

In [ ]:
#| code-fold: true
#| fig-cap: "The optimal discriminator for $p_{\\text{data}} = \\mathcal{N}(0, 1)$ and $p_g = \\mathcal{N}(1.5, 1)$. $D^*$ is a sigmoid of $x$ here (the log density ratio of two equal-variance Gaussians is linear), crossing $\\tfrac{1}{2}$ where the densities are equal."
#| fig-alt: "Two bell curves, p_data centred at 0 and p_g centred at 1.5, and a decreasing S-shaped curve D-star(x) going from near 1 on the left to near 0 on the right, passing through 0.5 at x = 0.75 where the two densities cross."
import numpy as np
import matplotlib.pyplot as plt

def log_normal_pdf(x, mean, sd):
    return -0.5 * np.log(2 * np.pi * sd**2) - (x - mean) ** 2 / (2 * sd**2)

x = np.linspace(-12, 14, 26001)
dx = x[1] - x[0]

def integrate(y):
    return np.sum(y) * dx

def kl(p, q):                        # D_KL[p || q], with 0 log 0 = 0
    ratio = np.divide(p, q, out=np.ones_like(p), where=p > 0)
    return integrate(p * np.log(ratio))

def js(p, q):
    m = (p + q) / 2
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

def value(f, p_data, p_g):
    """V(D) for D = sig(f): E_data[log D] + E_g[log(1 - D)], computed stably."""
    return integrate(-p_data * np.logaddexp(0, -f) - p_g * np.logaddexp(0, f))

log_p_data = log_normal_pdf(x, 0.0, 1.0)
theta = 1.5
log_p_g = log_normal_pdf(x, theta, 1.0)
p_data, p_g = np.exp(log_p_data), np.exp(log_p_g)
f_star = log_p_data - log_p_g        # D* = p_data / (p_data + p_g) = sig(f_star)

print(f"theta = {theta}:  V(D*) = {value(f_star, p_data, p_g):.6f},  "
      f"-log 4 + 2 JS = {-np.log(4) + 2 * js(p_data, p_g):.6f}")
for name, f in [("D = 1/2 everywhere", np.zeros_like(x)),
                ("D* logit - 0.5", f_star - 0.5),
                ("D* logit x 2", 2 * f_star)]:
    print(f"   {name:20}: V(D) = {value(f, p_data, p_g):.6f}  (worse)")

print("\ngenerator objective C(G) = V(D*) as the generator moves onto the data:")
for th in [4.0, 2.0, 1.0, 0.5, 0.0]:
    log_pg = log_normal_pdf(x, th, 1.0)
    C = value(log_p_data - log_pg, p_data, np.exp(log_pg))
    print(f"   theta = {th:3.1f}:  JS = {js(p_data, np.exp(log_pg)):.4f},  C(G) = {C:.4f}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x, p_data, label=r"$p_{\mathrm{data}}$")
ax.plot(x, p_g, label=r"$p_g$")
ax.plot(x, 1 / (1 + np.exp(-f_star)), "k", label=r"$D^*(x)$")
ax.axhline(0.5, color="gray", lw=0.8, ls=":")
ax.set_xlim(-4, 5.5)
ax.set_xlabel("$x$")
ax.legend()
plt.show()

## Common failures: mode dropping and collapse

**Mode dropping**: the generator represents only a subset of the training
distribution (low *coverage*).


**Mode collapse**: the extreme case where the generator mostly ignores
$\mathbf{z}$ and collapses all samples to a few points.


![UDL ch. 15.](figs/17-VAEs-and-GANs/mode-collapse-samples.png)

Nothing in the loss rewards covering *every* real sample: the generator is
only ever asked whether its samples look real, not whether all real samples
look generated. Compare the VAE, whose likelihood objective penalizes every
real example that the model assigns low probability.

## JS versus Wasserstein distance

**Jensen-Shannon** (UDL C.5.2) is a symmetrized KL:

$$
D_{JS}\big[p \,\|\, q\big] = \tfrac{1}{2} D_{KL}\Big[p \,\Big\|\, \tfrac{p + q}{2}\Big] + \tfrac{1}{2} D_{KL}\Big[q \,\Big\|\, \tfrac{p + q}{2}\Big]
$$

In $D_{JS}[p_g \,\|\, p_{\text{data}}]$ the first term rewards *quality*
(generated samples where real ones are) and the second *coverage* (real samples
where generated ones are) (UDL §15.2.1).

**The problem**: when the two distributions do not overlap, which is typical
early in training, $D_{JS}$ is a constant $\log 2$ and its gradient is zero.
The discriminator becomes too good and the generator stops learning.


**Wasserstein (earth mover's) distance**: the work needed to move the mass of
one distribution into the shape of the other, through an optimal transport plan
$\mathbf{P}$ that minimizes $\sum \mathbf{P} \cdot |i - j|$.

![UDL ch. 15.](figs/17-VAEs-and-GANs/wasserstein-transport-plan.png)

It grows smoothly with the *distance* between the distributions even when they
do not overlap, so it always gives a useful gradient. The Wasserstein GAN
([Arjovsky et al., 2017](https://arxiv.org/abs/1701.07875)) trains the
discriminator as a critic that estimates it. Details and
[UDL notebook 15.2](https://github.com/udlbook/udlbook/blob/main/Notebooks/Chap15/15_2_Wasserstein_Distance.ipynb)
in module M2.


**See the flat gradient.** Let the data be uniform on $[0, 1]$ and the
generator uniform on $[\theta, \theta + 1]$. Once $\theta \geq 1$ the supports
are disjoint, $D^*$ is 1 on the data and 0 on the samples, and $D_{JS}$ is
exactly $\log 2$ however far apart they are, so $C(G) = -\log 4 + 2\log 2 = 0$
and its gradient in $\theta$ is zero. The Wasserstein-1 distance between two
copies of the same shape shifted by $\theta$ is just $|\theta|$, which keeps
pointing the generator home.

In [ ]:
#| code-fold: true
#| fig-cap: "Jensen-Shannon divergence and Wasserstein-1 distance between $U[0, 1]$ and $U[\\theta, \\theta + 1]$, by numerical integration. JS saturates at $\\log 2$ once the supports stop overlapping ($\\theta \\geq 1$); the Wasserstein distance keeps growing linearly."
#| fig-alt: "Two curves against the shift theta from 0 to 3. The JS divergence rises linearly from 0 and flattens at log 2 = 0.693 for theta of 1 and beyond, marked by a dashed line. The Wasserstein-1 distance is a straight line from 0 to 3."
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-1, 5, 60001)
dx = x[1] - x[0]

def uniform(lo, hi):
    return ((x >= lo) & (x < hi)) / (hi - lo)

def kl(p, q):                         # D_KL[p || q], with 0 log 0 = 0
    ratio = np.divide(p, q, out=np.ones_like(p), where=p > 0)
    return np.sum(p * np.log(ratio)) * dx

def js(p, q):
    m = (p + q) / 2
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

def wasserstein1(p, q):               # 1D: integral of |CDF_p - CDF_q|
    return np.sum(np.abs(np.cumsum(p) - np.cumsum(q)) * dx) * dx

p_data = uniform(0, 1)
thetas = np.linspace(0, 3, 61)
js_vals = [js(p_data, uniform(t, t + 1)) for t in thetas]
w_vals = [wasserstein1(p_data, uniform(t, t + 1)) for t in thetas]

for t in [0.5, 1.0, 2.0, 3.0]:
    p_g = uniform(t, t + 1)
    print(f"theta = {t:.1f}:  JS = {js(p_data, p_g):.4f} (log 2 = {np.log(2):.4f}),  "
          f"W1 = {wasserstein1(p_data, p_g):.4f}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(thetas, js_vals, label=r"$D_{JS}$")
ax.plot(thetas, w_vals, label=r"Wasserstein-1")
ax.axhline(np.log(2), color="gray", ls="--", lw=0.8, label=r"$\log 2$")
ax.set_xlabel(r"generator shift $\theta$")
ax.legend()
plt.show()

## Conditional GANs

![UDL ch. 15.](figs/17-VAEs-and-GANs/conditional-gan.png)

- A vanilla GAN gives no control over *what* it generates.
- A conditional GAN ([Mirza & Osindero, 2014](https://arxiv.org/abs/1411.1784))
  passes an attribute vector $\mathbf{c}$ to both the generator and the
  discriminator; the discriminator judges whether the *(sample, attribute) pair*
  looks real, so the generator must respect the attribute.
- $\mathbf{c}$ can be a class label, a one-hot vector, real-valued attributes
  (smiling, male, wearing hat), a text embedding, a segmentation map, or an
  image. The same device conditions diffusion models on text in lecture 18.
- Variants that learn the attributes instead of being given them (auxiliary
  classifier GAN, InfoGAN) are in module M2.

## Image translation with paired data: pix2pix

![UDL ch. 15.](figs/17-VAEs-and-GANs/pix2pix.png)

- The conditioning variable is an entire image $\mathbf{c}$ (a sketch, a map,
  a grayscale photo); the generator is a U-Net (lecture 8) that maps it to the
  target image.
- Two losses: a **content loss** ($\ell_1$ norm) that makes the *prediction*
  similar to the ground truth, and an **adversarial loss** from a discriminator
  that sees (input, output) pairs and asks whether the pair looks real.
- Needs *paired* training data: each input with its matching target.

[Isola et al., "Image-to-Image Translation with Conditional Adversarial Networks", 2016](https://arxiv.org/abs/1611.07004)

## Image translation without pairs: CycleGAN

![UDL ch. 15.](figs/17-VAEs-and-GANs/cyclegan.png)

![Zhu et al., 2017.](figs/17-VAEs-and-GANs/cyclegan-results.png)


Two sets of images with distinct styles but **no matching pairs**: horses and
zebras, photos and Monet paintings.

Train a forward generator and an inverse one, with three losses:

1. **Adversarial** loss: the translated image looks like the target domain
2. **Content** loss ($\ell_1$): the translation stays close to the input
3. **Cycle-consistency** loss: translating forward then back returns the
   original

The cycle loss is what replaces the pairs: it forces the generator to be
(approximately) invertible, so it cannot throw away the input's content.

[Zhu et al., "Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks", 2017](https://arxiv.org/abs/1703.10593)


## Adversarial losses as a tool today

Pure GANs have mostly been replaced by diffusion and flow models as
*generators* (lecture 18). The **adversarial loss** survived as a training
tool wherever a model must produce sharp, realistic output and a likelihood
would average over details:

- **The autoencoders under latent diffusion.** VQGAN
  ([Esser et al., 2021](https://arxiv.org/abs/2012.09841)) and the KL
  autoencoder of Stable Diffusion
  ([Rombach et al., 2022](https://arxiv.org/abs/2112.10752)) add a patch-based
  discriminator to the reconstruction loss. Without it, decoded images are
  blurry, as in the plain VAE.
- **Few-step distillation.** Adversarial Diffusion Distillation
  ([Sauer et al., 2023](https://arxiv.org/abs/2311.17042), SDXL-Turbo) and its
  latent version ([Sauer et al., 2024](https://arxiv.org/abs/2403.12015)) use a
  discriminator to train a 1 to 4 step student from a diffusion teacher.
- **Audio.** Neural vocoders such as HiFi-GAN
  ([Kong et al., 2020](https://arxiv.org/abs/2010.05646)) turn spectrograms into
  waveforms with adversarial training, and the codecs under speech LLMs do the
  same.
- **GANs are not dead.** GigaGAN
  ([Kang et al., 2023](https://arxiv.org/abs/2303.05511)) scales a text-to-image
  GAN to about a billion parameters, and R3GAN
  ([Huang et al., 2025](https://arxiv.org/abs/2501.05441)) shows a modern
  baseline that trains stably without the usual tricks. Their appeal is a single
  forward pass per sample.


The pattern in every case: a reconstruction or regression loss ($\ell_1$, $\ell_2$,
perceptual) provides a stable training signal that captures the *content*, and
the adversarial term supplies the *texture* that an averaging loss would blur.
Module M2 covers the GAN architectures and tricks (DCGAN, progressive growing,
truncation, StyleGAN) for anyone whose project uses one.


# Summary

One line per idea; the recap deck uses these.


## Summary

- A generative model learns $\mathrm{Pr}(\mathbf{x})$ from samples; a *probabilistic* model can score an example, a *latent-variable* model maps $\mathbf{z} \to \mathbf{x}$. VAEs are both; GANs are only the second.
- An autoencoder minimizes $\|x - d(e(x))\|^2$; a linear one spans the PCA subspace. Sampling its latent space gives junk because nothing regularizes the space between codes.
- A VAE encodes $x$ as $\mathcal{N}(\mu_x, \sigma_x^2)$ and adds $\mathrm{KL}[\mathcal{N}(\mu_x, \sigma_x^2) \,\|\, \mathcal{N}(0, I)]$ to the reconstruction loss, which makes the latent space continuous and complete.
- ELBO: $\log p(x) = \mathbb{E}_{q_x}[\log p(x \mid z)] - KL(q_x \,\|\, p(z)) + KL(q_x \,\|\, p(z \mid x))$; the last term is $\geq 0$, so the first two are a lower bound on the evidence. $\log p(x)$ is constant in the encoder, so maximizing the ELBO over $(g, h)$ minimizes the gap.
- With $p(x \mid z) = \mathcal{N}(f(z), cI)$ the expected log likelihood is $-\|x - f(z)\|^2 / 2c$: the reconstruction term, with $c$ setting the trade-off against the KL.
- For diagonal Gaussians $KL = \tfrac{1}{2}\sum_d (\sigma_d^2 + \mu_d^2 - 1 - \log \sigma_d^2)$, computed in closed form; the reconstruction term is a one-sample Monte-Carlo estimate.
- Reparameterization: $z = \mu_x + \sigma_x \odot \zeta$, $\zeta \sim \mathcal{N}(0, I)$, makes $z$ a differentiable function of the encoder; gradients flow through both $\mu_x$ and $\sigma_x$, with $\zeta$ fixed for the step.
- GAN: generator $\mathbf{g}[\mathbf{z}, \boldsymbol{\theta}]$ and discriminator $\mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]$ play a minimax game on the cross-entropy loss; the optimum is a saddle point, so training is unstable.
- The optimal discriminator is the density ratio $D^* = p_{\text{data}} / (p_{\text{data}} + p_g)$, and at $D^*$ the generator minimizes $2 D_{JS}[p_{\text{data}} \,\|\, p_g] - \log 4$.
- Mode collapse: the loss rewards realism, not coverage. JS has zero gradient when supports do not overlap; Wasserstein distance does not.
- Conditional GANs feed $\mathbf{c}$ to both networks; pix2pix translates paired images with $\ell_1$ + adversarial loss; CycleGAN drops the pairs with a cycle-consistency loss.
- Today the adversarial loss is a tool: it sharpens the VAE under latent diffusion, distills few-step samplers, and trains vocoders.

Lecture 18 builds a diffusion model as a latent-variable model with a thousand
latents ($T = 1000$) and a *fixed* encoder that adds noise: its loss is this lecture's ELBO,
and every forward step is a reparameterized Gaussian sample. Module M2 has the
GAN architectures and tricks in depth.
